[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/17_sequence_and_state_space.ipynb)

# 17. Sequence recurrence and state-space updates — from SSM to Mamba

이전 버전의 `gate*h + (1-gate)*x`는 Mamba의 selective SSM이 아니었다.

여기서는 **continuous SSM → discretization → scan → input-dependent Δ/B/C → Mamba-style block** 순서로 실제 핵심 계산 그래프를 복구한다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Discrete recurrent state update

가장 기본적인 linear state-space recurrence는 `h_t = A_bar h_{t-1} + B_bar x_t`, `y_t = C h_t + D x_t` 형태다. Transformer attention처럼 모든 과거 token과 직접 score matrix를 만들지 않고 fixed-size state를 순서대로 갱신한다.


In [ ]:
inputs = torch.tensor(
    [1.0, 0.0, -1.0, 0.5],
    device=device,
)

A_bar = torch.tensor(
    [[0.8, 0.1], [0.0, 0.9]],
    device=device,
)
B_bar = torch.tensor(
    [1.0, 0.5],
    device=device,
)
C = torch.tensor(
    [0.7, -0.2],
    device=device,
)
D = torch.tensor(0.1, device=device)

state = torch.zeros(2, device=device)
outputs = []

for x_t in inputs:
    state = A_bar @ state + B_bar * x_t
    y_t = C @ state + D * x_t
    outputs.append(y_t)

print("outputs:", torch.stack(outputs))


## 2. Continuous SSM and discretization

S4/Mamba 계열의 출발점은 continuous state equation `dh/dt = A h + B x`다. time step `Δ`를 정하면 `A_bar = exp(ΔA)` 같은 discrete transition으로 바꿔 scan할 수 있다.

아래에서는 이해를 위해 diagonal `A`를 사용한다. Mamba의 CUDA reference도 핵심적으로 `deltaA = exp(delta * A)`를 계산한다.


In [ ]:
A = torch.tensor(
    [-1.0, -2.0],
    device=device,
)
B = torch.tensor(
    [1.0, 0.5],
    device=device,
)
delta = torch.tensor(0.25, device=device)

A_discrete = torch.exp(delta * A)
B_input = delta * B

print("exp(delta*A):", A_discrete)
print("delta*B:", B_input)


## 3. Associative scan viewpoint

각 step의 affine map `h ↦ a_t h + b_t`는 `(a2,b2)∘(a1,b1)=(a2*a1, a2*b1+b2)`로 결합할 수 있다. 이 associative structure 때문에 recurrence를 GPU scan으로 병렬화할 수 있다.


In [ ]:
a1 = torch.tensor(0.5, device=device)
b1 = torch.tensor(1.0, device=device)
a2 = torch.tensor(0.8, device=device)
b2 = torch.tensor(2.0, device=device)

a_composed = a2 * a1
b_composed = a2 * b1 + b2

h0 = torch.tensor(3.0, device=device)
sequential = a2 * (a1 * h0 + b1) + b2
composed = a_composed * h0 + b_composed

print("sequential:", sequential)
print("composed:", composed)


## 4. Mamba selective scan: Δ, B, C depend on the input

Mamba의 핵심은 단순 gate가 아니라 **각 token에서 Δ_t, B_t, C_t를 입력으로부터 생성**한다는 점이다. 그래서 무엇을 state에 넣고, 얼마나 오래 유지하고, 어떻게 읽을지가 token마다 달라진다.

reference selective scan의 핵심 recurrence를 작은 shape로 직접 구현한다.


In [ ]:
def selective_scan_reference(
    u,
    delta,
    A,
    B,
    C,
    D=None,
):
    batch_size, channels, sequence_length = u.shape
    state_dim = A.size(1)

    state = torch.zeros(
        batch_size,
        channels,
        state_dim,
        device=u.device,
    )
    outputs = []

    delta_A = torch.exp(
        delta[:, :, :, None]
        * A[None, :, None, :]
    )

    for time_index in range(sequence_length):
        delta_t = delta[:, :, time_index]
        u_t = u[:, :, time_index]
        B_t = B[:, time_index, :]
        C_t = C[:, time_index, :]

        input_to_state = (
            delta_t[:, :, None]
            * B_t[:, None, :]
            * u_t[:, :, None]
        )

        state = (
            delta_A[:, :, time_index, :] * state
            + input_to_state
        )

        y_t = torch.sum(
            state * C_t[:, None, :],
            dim=-1,
        )

        if D is not None:
            y_t = y_t + D[None, :] * u_t

        outputs.append(y_t)

    return torch.stack(outputs, dim=-1)


batch_size = 1
channels = 3
sequence_length = 5
state_dim = 2

u = torch.randn(
    batch_size,
    channels,
    sequence_length,
    device=device,
)
raw_features = u.transpose(1, 2)

delta_projection = nn.Linear(channels, channels).to(device)
B_projection = nn.Linear(channels, state_dim).to(device)
C_projection = nn.Linear(channels, state_dim).to(device)

delta = F.softplus(
    delta_projection(raw_features)
).transpose(1, 2)
B_variable = B_projection(raw_features)
C_variable = C_projection(raw_features)

A_log = nn.Parameter(
    torch.log(
        torch.arange(1, state_dim + 1, device=device)
        .float()
        .repeat(channels, 1)
    )
)
A = -torch.exp(A_log)
D = torch.ones(channels, device=device)

selective_output = selective_scan_reference(
    u,
    delta,
    A,
    B_variable,
    C_variable,
    D,
)

print("delta shape:", delta.shape)
print("B_t shape:", B_variable.shape)
print("C_t shape:", C_variable.shape)
print("scan output:", selective_output.shape)


## 5. Tiny Mamba-style block

Mamba block은 selective scan만 덩그러니 쓰지 않는다. 입력 projection을 두 갈래로 나누고, 한 branch는 causal depthwise convolution 뒤 selective SSM으로 보내며, 다른 branch는 SiLU gate `z`로 사용한다. 마지막에 output projection으로 다시 model dimension으로 보낸다.


In [ ]:
class TinyMambaBlock(nn.Module):
    def __init__(self, model_dim=8, inner_dim=12, state_dim=4):
        super().__init__()

        self.inner_dim = inner_dim
        self.state_dim = state_dim

        self.input_projection = nn.Linear(
            model_dim,
            2 * inner_dim,
        )
        self.depthwise_conv = nn.Conv1d(
            inner_dim,
            inner_dim,
            kernel_size=3,
            padding=2,
            groups=inner_dim,
        )

        self.delta_projection = nn.Linear(inner_dim, inner_dim)
        self.B_projection = nn.Linear(inner_dim, state_dim)
        self.C_projection = nn.Linear(inner_dim, state_dim)

        initial_A = torch.arange(1, state_dim + 1).float()
        initial_A = initial_A.repeat(inner_dim, 1)
        self.A_log = nn.Parameter(torch.log(initial_A))
        self.D = nn.Parameter(torch.ones(inner_dim))

        self.output_projection = nn.Linear(inner_dim, model_dim)

    def forward(self, x):
        projected = self.input_projection(x)
        u_branch, z_branch = projected.chunk(2, dim=-1)

        u = u_branch.transpose(1, 2)
        convolved = self.depthwise_conv(u)
        convolved = convolved[:, :, : x.size(1)]
        convolved = F.silu(convolved)

        features = convolved.transpose(1, 2)
        delta = F.softplus(
            self.delta_projection(features)
        ).transpose(1, 2)
        B = self.B_projection(features)
        C = self.C_projection(features)

        A = -torch.exp(self.A_log)

        scanned = selective_scan_reference(
            convolved,
            delta,
            A,
            B,
            C,
            self.D,
        )

        gated = scanned.transpose(1, 2) * F.silu(z_branch)
        return self.output_projection(gated)


block = TinyMambaBlock().to(device)
sequence = torch.randn(
    2, 6, 8,
    device=device,
)
output = block(sequence)

print("input:", sequence.shape)
print("output:", output.shape)


## References and provenance

**S4** — Gu et al., *Efficiently Modeling Long Sequences with Structured State Spaces*. continuous/discrete state-space viewpoint와 scan 구조를 참조했다.

**Mamba** — Gu & Dao, *Mamba: Linear-Time Sequence Modeling with Selective State Spaces* 및 공식 `state-spaces/mamba`의 `selective_scan_ref`. `deltaA=exp(delta*A)`, input-dependent `delta/B/C`, recurrent state update, `D*u`, input split, causal depthwise convolution, SiLU gate, output projection을 작은 형태로 보존했다.

KDA/DeltaNet은 matrix-valued recurrent state와 delta-rule update가 핵심이라 이 Mamba section과 같은 것으로 취급하지 않는다. 해당 계열은 09 advanced attention에서 별도로 다루는 것이 맞다.
